In [ ]:
import pandas as pd
import sqlite3
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
from sklearn.model_selection import train_test_split
import os
import shutil
import yaml
from ultralytics import YOLO
import cv2
import matplotlib.patches as patches
from pathlib import Path
import ipywidgets as widgets
from IPython.display import display, clear_output
from glob import glob
import tqdm

## Use this Notebook to convert new synthetic data into YOLO or CSRNet compatible data sets

In [ ]:
# --- Configuration ---
# This should be the main folder for your experiment
base_dir = '<DATASET_ROOT>/Exp_08_Polarized_OBB'
db_path = os.path.join(base_dir, 'stripGen_results.db')
output_dataset_dir = os.path.join(base_dir, 'dataset')

# --- 1. Load Data ---
print(f"Loading data from {db_path}...")
sqlite_connection = sqlite3.connect(db_path)
df = pd.read_sql_query("SELECT * FROM results", sqlite_connection)
sqlite_connection.close()

# Make sure image paths are absolute
df['image_path_aolp'] = df['image_path_aolp'].apply(lambda p: os.path.join(base_dir, p))
all_image_paths = df['image_path_aolp'].tolist()

In [ ]:
df.columns

In [ ]:
df.head(5)

In [ ]:
# Show a complete image path and its corresponding AOLP path, you can click on them to open
sample_index = 0
#print("Sample original image path:", df.loc[sample_index, 'image_path'])
print("Corresponding AOLP image path:", df.loc[sample_index, 'image_path_aolp'])

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='random_seed', y='strips_inside_box_count', data=df, palette='muted')
plt.xlabel('Seed')
plt.ylabel('Strips Inside Box Count')
plt.title('Distribution of Strips Inside Box Count for Each Seed')
plt.show()

In [ ]:
def visualize_yolo_bboxes(image_path, bbox_path, figsize=(12, 8)):
    """
    Visualize YOLO format bounding boxes on an image.
    
    Args:
        image_path: Path to the image file
        bbox_path: Path to the YOLO format text file
        figsize: Figure size for matplotlib
    """
    # Read image
    img = cv2.imread(str(image_path))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    height, width = img.shape[:2]
    
    # Create figure
    fig, ax = plt.subplots(1, figsize=figsize)
    ax.imshow(img)
    
    # Read and parse YOLO format bounding boxes
    if Path(bbox_path).exists():
        with open(bbox_path, 'r') as f:
            lines = f.readlines()
        
        for i, line in enumerate(lines):
            parts = line.strip().split()

            # Standard Bounding Box (ABB)
            if len(parts) == 5:
                class_id, cx, cy, w, h = map(float, parts)
                
                # Convert from YOLO format (normalized) to pixel coordinates
                # YOLO format: center_x, center_y, width, height (all normalized 0-1)
                pixel_cx = cx * width
                pixel_cy = cy * height
                pixel_w = w * width
                pixel_h = h * height
                
                # Convert center coordinates to top-left corner
                x1 = pixel_cx - pixel_w / 2
                y1 = pixel_cy - pixel_h / 2
                
                # Create rectangle patch
                rect = patches.Rectangle(
                    (x1, y1), pixel_w, pixel_h,
                    linewidth=2,
                    edgecolor='red',
                    facecolor='none',
                    alpha=0.7
                )
                ax.add_patch(rect)

            # Oriented Bounding Box (OBB) with angle    
            elif len(parts) == 6:
                class_id, cx, cy, w, h, angle = map(float, parts)
                
                # Convert from YOLO format (normalized) to pixel coordinates
                pixel_cx = cx * width
                pixel_cy = cy * height
                pixel_w = w * width
                pixel_h = h * height
                
                # Calculate corners for OBB
                # Unrotated corners relative to center
                corners = np.array([
                    [-pixel_w / 2, -pixel_h / 2],
                    [pixel_w / 2, -pixel_h / 2],
                    [pixel_w / 2, pixel_h / 2],
                    [-pixel_w / 2, pixel_h / 2]
                ])
                
                # Rotation matrix
                c, s = np.cos(angle), np.sin(angle)
                R = np.array(((c, -s), (s, c)))
                
                # Rotate and translate
                rotated_corners = np.dot(corners, R.T) + np.array([pixel_cx, pixel_cy])
                
                # Create polygon patch
                poly = patches.Polygon(
                    rotated_corners,
                    linewidth=2,
                    edgecolor='red',
                    facecolor='none',
                    alpha=0.7
                )
                ax.add_patch(poly)
                
                # Draw orientation arrow to verify angle
                # Points along the 'width' axis (which is the length of the strip)
                arrow_vec = np.dot(np.array([pixel_w/2, 0]), R.T)
                ax.arrow(pixel_cx, pixel_cy, arrow_vec[0], arrow_vec[1], 
                         head_width=min(pixel_w, pixel_h)*0.2, head_length=min(pixel_w, pixel_h)*0.2, 
                         fc='blue', ec='blue', alpha=0.5)
                
                # Add label with class id
                # ax.text(x1, y1 - 5, f'Strip {i}', 
                #        color='yellow', fontsize=8, 
                #        bbox=dict(facecolor='red', alpha=0.5))
        
        print(f"Drew {len(lines)} bounding boxes")
    else:
        print(f"No bbox file found at {bbox_path}")
    
    ax.set_title(f'Image: {Path(image_path).name}')
    ax.axis('off')
    plt.tight_layout()
    plt.show()
    
    #return fig

In [ ]:
def show_image_with_bboxes(idx):
    image_path = df['image_path_aolp'].iloc[idx]
    bbox_path = df['bbox_path'].iloc[idx]
    visualize_yolo_bboxes(image_path, bbox_path)

slider = widgets.IntSlider(value=0, min=0, max=len(df)-1, step=1, description='Image Index')
interactive_plot = widgets.interactive(show_image_with_bboxes, idx=slider)
display(interactive_plot)

##  Train/Test Split and File Copying

Only run this cell once, or when the database was modified

In [ ]:
# --- 1. Split Data ---
# We split the entire DataFrame to keep the association between aolp images and original paths.
train_df, val_df = train_test_split(df, test_size=0.2, random_state=42)
print(f"Data split: {len(train_df)} training images, {len(val_df)} validation images.")

# --- 2. Create Directories ---
for split in ['train', 'val']:
    os.makedirs(os.path.join(output_dataset_dir, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(output_dataset_dir, 'labels', split), exist_ok=True)

# --- 3. Copy Files ---
def copy_files_notebook(dataframe_split, split_name):
    """Copies AOLP images and their corresponding labels to the dataset folder."""
    for _, row in dataframe_split.iterrows():
        # Source paths
        aolp_image_path = row['image_path_aolp']
        original_image_path = row['image_path']
        
        # The label file is named after the original image
        label_path = os.path.splitext(original_image_path)[0] + '.txt'
        
        # Destination paths
        dest_img_path = os.path.join(output_dataset_dir, 'images', split_name, os.path.basename(aolp_image_path))
        
        # The destination label should have the same name as the AOLP image, but with a .txt extension
        dest_label_filename = os.path.splitext(os.path.basename(aolp_image_path))[0] + '.txt'
        dest_label_path = os.path.join(output_dataset_dir, 'labels', split_name, dest_label_filename)
        
        if os.path.exists(aolp_image_path) and os.path.exists(label_path):
            shutil.copy(aolp_image_path, dest_img_path)
            shutil.copy(label_path, dest_label_path)

print("Copying AOLP images and labels for training and validation sets...")
copy_files_notebook(train_df, 'train')
copy_files_notebook(val_df, 'val')

print(f"✅ Dataset preparation complete! Files are organized in '{output_dataset_dir}'.")

## Define YOLO Config

In [ ]:

# Define the dataset configuration
yaml_content = {
    'path': output_dataset_dir,  # Root dataset directory
    'train': 'images/train',     # Path to training images (relative to 'path')
    'val': 'images/val',         # Path to validation images (relative to 'path')
    'nc': 1,                     # Number of classes
    'names': ['strip']           # List of class names
}

# Write the configuration to a data.yaml file
yaml_path = os.path.join(output_dataset_dir, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(yaml_content, f, sort_keys=False)

print(f"✅ YAML configuration file created at: {yaml_path}")

## Train Model

run the batch skript on the HPC Cluster, copy the whole experiment folder!

### MAKE SURE TO UPDATE THE 'path': output_dataset_dir,  in the yaml file on the cluster to point to the correct directory

In [ ]:
# # Load a pre-trained YOLOv8n (nano) model
# model = YOLO('yolo11n.pt')

# # Define project and run name for storing results
# # The results will be saved to project/name
# project_dir = os.path.join(base_dir, 'runs/detect')
# run_name = 'train'

# # Train the model using your YAML file
# results = model.train(data=yaml_path, epochs=25, imgsz=512, project=project_dir, name=run_name)

# # The results object contains the path to the directory where results are saved
# # We can use this path in other cells
# results_dir = results.save_dir
# print(f"Results saved to: {results_dir}")

In [ ]:
# Load results from csv
results_path = '<DATASET_ROOT>/Exp_07_Polarized/runs/detect/train/results.csv'
if os.path.exists(results_path):
    results_df = pd.read_csv(results_path)
    # The column names can have leading/trailing spaces, let's strip them
    results_df.columns = results_df.columns.str.strip()
    print("YOLO training results loaded successfully.")
    display(results_df.head())
else:
    print(f"Could not find results file at: {results_path}")

## Point Annotations for CSRNet

In [ ]:
# --- Convert Bounding Boxes to Point Annotations for Train and Val ---

# This cell converts YOLO bounding box annotations from both the training and
# validation sets into point annotations (center of the box), suitable for
# density estimation models like CSRNet.

if 'base_dir' in locals():
    print("Starting conversion from Bounding Box to Point Annotations for train and val sets...")
    
    splits = ['train', 'val']
    
    for split in splits:
        print(f"\n--- Processing split: {split} ---")
        
        # --- Configuration for the current split ---
        labels_source_dir = os.path.join(base_dir, 'dataset', 'labels', split)
        images_source_dir = os.path.join(base_dir, 'dataset', 'images', split)
        output_points_dir = os.path.join(base_dir, 'dataset', 'points_annotations', split)

        # Check if source directory exists
        if not os.path.isdir(labels_source_dir):
            print(f"Source directory not found: {labels_source_dir}. Skipping.")
            continue

        # Create the output directory if it doesn't exist
        os.makedirs(output_points_dir, exist_ok=True)
        
        # Get a list of all label files
        label_files = [f for f in os.listdir(labels_source_dir) if f.endswith('.txt')]
        
        if not label_files:
            print("No label files found to convert.")
            continue
            
        converted_count = 0
        
        # --- Processing Loop ---
        for label_file in tqdm.tqdm(label_files, desc=f"Converting {split} annotations"):
            label_path = os.path.join(labels_source_dir, label_file)
            
            # Find the corresponding image file (.png or .jpg)
            image_name_base = os.path.splitext(label_file)[0]
            image_path_png = os.path.join(images_source_dir, image_name_base + '.png')
            image_path_jpg = os.path.join(images_source_dir, image_name_base + '.jpg')
            
            image_path = None
            if os.path.exists(image_path_png):
                image_path = image_path_png
            elif os.path.exists(image_path_jpg):
                image_path = image_path_jpg
            else:
                # This warning can be noisy if there are non-image files, so it's commented out.
                # print(f"  - Warning: Could not find image for label {label_file}. Skipping.")
                continue
                
            # Get image dimensions (height, width)
            try:
                img = cv2.imread(image_path)
                if img is None:
                    # print(f"  - Warning: Failed to read image {image_path}. Skipping.")
                    continue
                img_height, img_width, _ = img.shape
            except Exception as e:
                # print(f"  - Error reading image {image_path}: {e}. Skipping.")
                continue

            # --- Read YOLO annotations and convert ---
            points = []
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        # YOLO format: <class> <x_center_norm> <y_center_norm> <width_norm> <height_norm> [angle]
                        x_center_norm = float(parts[1])
                        y_center_norm = float(parts[2])
                        
                        # Convert normalized coordinates to absolute pixel coordinates
                        x_abs = int(x_center_norm * img_width)
                        y_abs = int(y_center_norm * img_height)
                        
                        points.append([x_abs, y_abs])
            
            # --- Save the new point annotations ---
            if points:
                output_filename = os.path.join(output_points_dir, image_name_base + '.csv')
                points_df = pd.DataFrame(points, columns=['x', 'y'])
                points_df.to_csv(output_filename, index=False)
                converted_count += 1

        print(f"Conversion for '{split}' split complete. Successfully converted {converted_count}/{len(label_files)} annotation files.")
        print(f"Point annotations are saved in: {output_points_dir}")

    print("\n✅ All splits processed.")
else:
    print("Please run the configuration cell first to define 'base_dir'.")

In [ ]:
# --- Visualize Point Annotations on a Random Image ---

# This cell loads a random image from the validation set and overlays the
# generated point annotations to verify the conversion process.

if 'base_dir' in locals():
    print("Visualizing point annotations on a random validation image...")
    
    # --- Configuration ---
    split_to_visualize = 'val'
    images_dir = os.path.join(base_dir, 'dataset', 'images', split_to_visualize)
    points_dir = os.path.join(base_dir, 'dataset', 'points_annotations', split_to_visualize)
    
    # --- Select a Random Image ---
    if not os.path.isdir(images_dir):
        print(f"Images directory not found: {images_dir}")
    else:
        all_images = [f for f in os.listdir(images_dir) if f.endswith(('.png', '.jpg'))]
        if not all_images:
            print("No images found in the directory.")
        else:
            random_image_name = np.random.choice(all_images)
            image_path = os.path.join(images_dir, random_image_name)
            
            # Corresponding points file
            points_filename = os.path.splitext(random_image_name)[0] + '.csv'
            points_path = os.path.join(points_dir, points_filename)
            
            # --- Load and Draw ---
            if os.path.exists(image_path) and os.path.exists(points_path):
                # Load image
                img = cv2.imread(image_path)
                img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
                
                # Load points
                points_df = pd.read_csv(points_path)
                
                # Draw a circle for each point
                for index, row in points_df.iterrows():
                    center_coordinates = (int(row['x']), int(row['y']))
                    cv2.circle(img, center_coordinates, radius=3, color=(255, 0, 0), thickness=-1) # Red filled circle
                
                # --- Display ---
                plt.figure(figsize=(15, 10))
                plt.imshow(img)
                plt.title(f'Image: {random_image_name}\n{len(points_df)} Points Annotated')
                plt.axis('off')
                plt.show()
                
            else:
                print(f"Could not find matching image/annotation for {random_image_name}")
else:
    print("Please run the configuration cell first to define 'base_dir'.")


In [ ]:
# --- Generate and Visualize a Ground Truth Density Map ---

# This cell explains and demonstrates the final step required before training a
# density estimation model like CSRNet: converting point annotations into a
# ground truth density map.

import scipy.ndimage
import scipy.io

def generate_density_map(image_shape, points, sigma=4):
    """
    Generates a density map from a list of points.
    
    Args:
        image_shape (tuple): The (height, width) of the image.
        points (np.array): An array of shape (n, 2) with [x, y] coordinates.
        sigma (float): The standard deviation for the Gaussian kernel. This controls
                       how "spread out" each point is in the density map.
    
    Returns:
        np.array: The generated density map.
    """
    height, width = image_shape[:2]
    density_map = np.zeros((height, width), dtype=np.float32)
    
    for x, y in points:
        # Ensure the point is within the image boundaries
        if 0 <= x < width and 0 <= y < height:
            density_map[int(y), int(x)] = 1
            
    # Convolve with a Gaussian kernel to create the density map
    density_map = scipy.ndimage.gaussian_filter(density_map, sigma=sigma, mode='constant')
    
    return density_map

if 'base_dir' in locals():
    print("Visualizing a sample density map...")
    
    # --- Use the same random image from the previous cell ---
    if 'image_path' in locals() and os.path.exists(image_path):
        
        # Load the image and its points
        img_for_density = cv2.imread(image_path)
        points_df = pd.read_csv(points_path)
        
        # Generate the density map
        # Note: CSRNet often downsamples the output, but for visualization,
        # we create a full-size map here.
        density_map = generate_density_map(img_for_density.shape, points_df[['x', 'y']].values)
        
        # --- Display the results ---
        fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(24, 8))
        
        # Original Image with points
        img_rgb = cv2.cvtColor(img_for_density, cv2.COLOR_BGR2RGB)
        for index, row in points_df.iterrows():
            cv2.circle(img_rgb, (int(row['x']), int(row['y'])), radius=3, color=(255, 0, 0), thickness=-1)
        ax1.imshow(img_rgb)
        ax1.set_title(f'Original Image with {len(points_df)} Points')
        ax1.axis('off')
        
        # Density Map
        im = ax2.imshow(density_map, cmap='jet')
        ax2.set_title('Generated Density Map')
        ax2.axis('off')
        fig.colorbar(im, ax=ax2)
        
        # Overlay
        ax3.imshow(img_rgb)
        ax3.imshow(density_map, cmap='jet', alpha=0.5)
        ax3.set_title('Density Map Overlay')
        ax3.axis('off')
        
        plt.tight_layout()
        plt.show()
        
        # Verification: The sum of the density map should be close to the point count
        print(f"Number of points: {len(points_df)}")
        print(f"Sum of density map: {np.sum(density_map):.2f}")
        
    else:
        print("Please run the previous cell first to select a random image.")
else:
    print("Please run the configuration cell first to define 'base_dir'.")
